IntelliRAG — Tam Pipeline

---

**Runtime → GPU'yu Etkinleştirin:** `Runtime > Change runtime type > T4 GPU`

**1. Kurulum**

In [ ]:
# ── Tüm bağımlılıkları yükleme ─────────────────────────────────────────────────
!pip install -q \
    transformers==4.40.0 \
    sentence-transformers==2.7.0 \
    faiss-cpu==1.8.0 \
    rank-bm25==0.2.2 \
    bitsandbytes==0.43.1 \
    datasets==2.19.0 \
    ragas==0.1.7 \
    numpy==1.26.4 \
    tqdm rich

print("Kurulum tamamlandı!")

In [ ]:
# ── Import'lar ───────────────────────────────────────────────────────────────
import os
import re
import time
import json
import warnings
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass, field
from tqdm.notebook import tqdm

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss

warnings.filterwarnings('ignore')

# GPU kontrolü
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Cihaz: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Üretkenlik için seed
torch.manual_seed(42)
np.random.seed(42)

##2. Veri Seti — Natural Questions (NQ)

Google'ın hazırladığı **Natural Questions** veri setini kullanıldı.  
Kısmi değerlendirme 100 örnek alındı; tam değerlendirme için `n_samples=7830` seçmeliyiz.

In [ ]:
# ── NQ veri setini yükle ─────────────────────────────────────────────────────
print("Natural Questions veri seti yükleniyor...")

dataset = load_dataset(
    "natural_questions",
    split="validation",
    trust_remote_code=True
)

N_SAMPLES = 100  # Tam tez için: 7830
dataset = dataset.select(range(N_SAMPLES))

print(f"{len(dataset)} soru yüklendi")
print(f"\n Örnek soru:")
print(f"   Soru : {dataset[0]['question']['text']}")

# Ham veriyi düzenle
samples = []
for item in dataset:
    question = item['question']['text']
    context_text = item['document']['tokens']['token']
    context = ' '.join(context_text[:500])  # İlk 500 token

    # Kısa cevap al
    annotations = item['annotations']
    answer = ""
    for ann in annotations['short_answers']:
        if ann['text']:
            answer = ann['text'][0]
            break

    if answer:  # Sadece cevabı olanları al
        samples.append({
            'question': question,
            'context': context,
            'answer': answer
        })

print(f"\n Kullanılabilir örnek sayısı: {len(samples)}")
print(f"\n--- Örnek 1 ---")
print(f"Soru  : {samples[0]['question']}")
print(f"Cevap : {samples[0]['answer']}")
print(f"Bağlam: {samples[0]['context'][:200]}...")

## Modül 1 — Dinamik Anlamsal Parçalama

Belgeleri **sabit boyut** yerine **anlam sınırlarından** ayırdık.  
Benzerlik eşiği altına düşen noktalarda yeni parça başlatılıyor.

In [ ]:
# ── Modül 1: SemanticChunker ──────────────────────────────────────────────────
@dataclass
class SemanticChunker:
    """
    Anlam benzerliğine dayalı dinamik belge parçalayıcı.
    Cümle embedding'leri arasındaki kosinüs benzerliği
    eşik değerinin altına düştüğünde yeni parça açılır.
    """
    model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2'
    similarity_threshold: float = 0.75  # Bu değerin altı → yeni parça
    min_chunk_size: int = 50            # Minimum token sayısı
    max_chunk_size: int = 512           # Maximum token sayısı

    def __post_init__(self):
        print(f"   Model yükleniyor: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"    SemanticChunker hazır")

    def _split_sentences(self, text: str) -> List[str]:
        """Metni cümlelere böl."""
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 10]

    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """İki vektör arasındaki kosinüs benzerliği."""
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

    def chunk(self, text: str) -> List[str]:
        """Metni semantik parçalara böl."""
        sentences = self._split_sentences(text)
        if not sentences:
            return [text]

        # Tüm cümleleri vektörleştir
        embeddings = self.model.encode(sentences, show_progress_bar=False)

        chunks = []
        current_chunk = [sentences[0]]
        current_tokens = len(sentences[0].split())

        for i in range(1, len(sentences)):
            sim = self._cosine_similarity(embeddings[i-1], embeddings[i])
            new_tokens = len(sentences[i].split())

            # Parça kesme koşulları
            should_split = (
                sim < self.similarity_threshold or                 # Benzerlik düşük
                current_tokens + new_tokens > self.max_chunk_size  # Çok büyük
            )

            if should_split and current_tokens >= self.min_chunk_size:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentences[i]]
                current_tokens = new_tokens
            else:
                current_chunk.append(sentences[i])
                current_tokens += new_tokens

        if current_chunk:
            chunks.append(' '.join(current_chunk))

        return chunks


# ── Test ──────────────────────────────────────────────────────────────────────
print("Modül 1: SemanticChunker başlatılıyor...")
chunker = SemanticChunker()

# Örnek belge üzerinde dene
sample_doc = samples[0]['context']
chunks = chunker.chunk(sample_doc)

print(f"\n Belge uzunluğu   : {len(sample_doc.split())} token")
print(f" Parça sayısı     : {len(chunks)}")
print(f" Ortalama parça   : {np.mean([len(c.split()) for c in chunks]):.0f} token")
print(f"\n--- İlk Parça ---")
print(chunks[0][:300] + "...")

## Modül 2 — Hibrit Geri Alma (BM25 + FAISS)

İki farklı arama yöntemini birleştirdik:
- **BM25** → Anahtar kelime eşleşmesi (seyrek)
- **FAISS** → Anlam benzerliği (yoğun)

In [ ]:
# ── Modül 2: HybridRetriever ─────────────────────────────────────────────────
@dataclass
class HybridRetriever:
    """
    BM25 (seyrek) + FAISS (yoğun) hibrit geri alma sistemi.
    Reciprocal Rank Fusion (RRF) ile sonuçları birleştirir.
    """
    encoder_model: str = 'paraphrase-multilingual-MiniLM-L12-v2'
    rrf_k: int = 60       # RRF sabiti (literatürde standart değer)
    bm25_weight: float = 0.4
    dense_weight: float = 0.6

    # Sonradan doldurulacak
    chunks: List[str] = field(default_factory=list)
    bm25_index: object = field(default=None, repr=False)
    faiss_index: object = field(default=None, repr=False)
    encoder: object = field(default=None, repr=False)

    def __post_init__(self):
        self.encoder = SentenceTransformer(self.encoder_model)
        print(f"    HybridRetriever hazır")

    def build_index(self, chunks: List[str]):
        """BM25 ve FAISS indekslerini oluştur."""
        self.chunks = chunks
        print(f"    {len(chunks)} parça indeksleniyor...")

        # BM25 indeksi
        tokenized = [c.lower().split() for c in chunks]
        self.bm25_index = BM25Okapi(tokenized)

        # FAISS indeksi
        embeddings = self.encoder.encode(
            chunks, show_progress_bar=False, convert_to_numpy=True
        ).astype('float32')

        faiss.normalize_L2(embeddings)
        dim = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dim)  # Inner Product (= kosinüs, normalize edilmiş)
        self.faiss_index.add(embeddings)
        self._embeddings = embeddings

        print(f"    BM25: {len(chunks)} dok | FAISS: {self.faiss_index.ntotal} vektör")

    def _rrf_score(self, bm25_ranks: Dict[int, int], dense_ranks: Dict[int, int]) -> Dict[int, float]:
        """Reciprocal Rank Fusion ile iki listeyi birleştir."""
        all_ids = set(bm25_ranks) | set(dense_ranks)
        scores = {}
        for idx in all_ids:
            bm25_r = bm25_ranks.get(idx, len(self.chunks) + 1)
            dense_r = dense_ranks.get(idx, len(self.chunks) + 1)
            scores[idx] = (
                self.bm25_weight / (self.rrf_k + bm25_r) +
                self.dense_weight / (self.rrf_k + dense_r)
            )
        return scores

    def retrieve(self, query: str, top_k: int = 10) -> List[Dict]:
        """Hibrit arama yap ve en iyi top_k sonucu döndür."""
        # BM25 arama
        bm25_scores = self.bm25_index.get_scores(query.lower().split())
        bm25_top = np.argsort(bm25_scores)[::-1][:top_k * 2]
        bm25_ranks = {int(idx): rank + 1 for rank, idx in enumerate(bm25_top)}

        # FAISS arama
        q_emb = self.encoder.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        _, dense_top = self.faiss_index.search(q_emb, top_k * 2)
        dense_ranks = {int(idx): rank + 1 for rank, idx in enumerate(dense_top[0]) if idx >= 0}

        # RRF birleştirme
        rrf_scores = self._rrf_score(bm25_ranks, dense_ranks)
        top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

        return [
            {'chunk': self.chunks[i], 'score': rrf_scores[i], 'id': i}
            for i in top_ids
        ]


# ── Tüm örnekleri indeksle ────────────────────────────────────────────────────
print("Modül 2: HybridRetriever başlatılıyor...")
retriever = HybridRetriever()

# Tüm bağlamları parçalara böl ve indeksle
print("\n Tüm belgeler parçalanıp indeksleniyor...")
all_chunks = []
chunk_to_doc = []  # Hangi parça hangi belgeden?

for doc_id, sample in enumerate(samples[:20]):  # Demo: ilk 20
    doc_chunks = chunker.chunk(sample['context'])
    all_chunks.extend(doc_chunks)
    chunk_to_doc.extend([doc_id] * len(doc_chunks))

retriever.build_index(all_chunks)

# Test sorgusu
test_query = samples[0]['question']
results = retriever.retrieve(test_query, top_k=5)

print(f"\n Sorgu: '{test_query}'")
print(f"\n İlk 3 sonuç:")
for i, r in enumerate(results[:3], 1):
    print(f"\n[{i}] Skor: {r['score']:.4f}")
    print(f"    {r['chunk'][:150]}...")

## Modül 3 — Çapraz Kodlayıcı Yeniden Sıralama

İlk arama sonuçlarını daha güçlü bir modelle yeniden puanladık.  
Cross-encoder, sorgu ve bağlamı **birlikte** değerlendirerek daha hassas sıralama yapar.

In [ ]:
# ── Modül 3: CrossEncoderReranker ────────────────────────────────────────────
@dataclass
class CrossEncoderReranker:
    """
    Çapraz kodlayıcı tabanlı yeniden sıralama modülü.
    Bi-encoder'ın hızlı ama yüzeysel sıralamasını derinleştirir.
    """
    model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
    top_k: int = 5  # Kaç sonuç döndürülsün? 5 tane dönsün

    def __post_init__(self):
        print(f"   Model yükleniyor: {self.model_name}")
        self.model = CrossEncoder(self.model_name)
        print(f"    CrossEncoderReranker hazır")

    def rerank(self, query: str, candidates: List[Dict]) -> List[Dict]:
        """
        Aday listesini cross-encoder puanlarına göre yeniden sırala.

        Args:
            query      : Kullanıcı sorusu
            candidates : HybridRetriever'dan gelen sonuçlar
        Returns:
            Yeniden sıralanmış ve top_k ile kırpılmış liste
        """
        if not candidates:
            return []

        # (sorgu, parça) çiftleri oluştur
        pairs = [(query, c['chunk']) for c in candidates]

        # Cross-encoder puanları
        ce_scores = self.model.predict(pairs)

        # Puanları ekle ve sırala
        for i, candidate in enumerate(candidates):
            candidate['ce_score'] = float(ce_scores[i])

        reranked = sorted(candidates, key=lambda x: x['ce_score'], reverse=True)
        return reranked[:self.top_k]


# ── Test ──────────────────────────────────────────────────────────────────────
print(" Modül 3: CrossEncoderReranker başlatılıyor...")
reranker = CrossEncoderReranker(top_k=3)

# Önceki arama sonuçlarını yeniden sırala
reranked_results = reranker.rerank(test_query, results)

print(f"\n Yeniden Sıralanmış Sonuçlar:")
print(f"   Sorgu: '{test_query}'\n")

for i, r in enumerate(reranked_results, 1):
    print(f"[{i}] CE Skoru: {r['ce_score']:.4f} | Hibrit Skoru: {r['score']:.4f}")
    print(f"    {r['chunk'][:150]}...\n")

## Modül 4 — Uyarlanabilir Bağlam Sıkıştırma (ACC)

Seçilen parçalardan **gereksiz cümleleri almadık**.  
Modele sadece en ilgili cümleleri veriyoruz → hem hız artar hem kalite.

In [ ]:
# ── Modül 4: AdaptiveContextCompressor ───────────────────────────────────────
@dataclass
class AdaptiveContextCompressor:
    """
    Seçilen bağlamdan gereksiz cümleleri eler.
    Sorgu ile yüksek benzerlikli cümleleri korur, düşükleri atar.
    """
    model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2'
    compression_ratio: float = 0.5  # Bağlamın %50'sini koru
    min_sentences: int = 2          # En az 2 cümle kalsın

    def __post_init__(self):
        self.model = SentenceTransformer(self.model_name)
        print(f"    AdaptiveContextCompressor hazır")

    def compress(self, query: str, contexts: List[str]) -> str:
        """
        Bağlamları sıkıştırıp tek bir metin döndür.

        Args:
            query    : Kullanıcı sorusu
            contexts : Yeniden sıralanmış bağlam listesi
        Returns:
            Sıkıştırılmış bağlam metni
        """
        # Sorgu vektörü
        q_emb = self.model.encode([query], convert_to_numpy=True)[0]

        all_sentences = []
        for ctx in contexts:
            sents = re.split(r'(?<=[.!?])\s+', ctx.strip())
            all_sentences.extend([s for s in sents if len(s.strip()) > 15])

        if not all_sentences:
            return ' '.join(contexts)

        # Cümle vektörleri
        s_embs = self.model.encode(all_sentences, convert_to_numpy=True)

        # Kosinüs benzerliği hesapla
        similarities = np.dot(s_embs, q_emb) / (
            np.linalg.norm(s_embs, axis=1) * np.linalg.norm(q_emb) + 1e-8
        )

        # Top-K cümle seç
        n_keep = max(
            self.min_sentences,
            int(len(all_sentences) * self.compression_ratio)
        )
        top_indices = np.argsort(similarities)[::-1][:n_keep]
        top_indices_sorted = sorted(top_indices)  # Özgün sırayı koru

        compressed = ' '.join([all_sentences[i] for i in top_indices_sorted])

        return compressed


# ── Test ──────────────────────────────────────────────────────────────────────
print("Modül 4: Uyarlanabilir Bağlam Sıkıştırma başlatılıyor...")
compressor = AdaptiveContextCompressor(compression_ratio=0.5)

raw_contexts = [r['chunk'] for r in reranked_results]
compressed = compressor.compress(test_query, raw_contexts)

raw_len = sum(len(c.split()) for c in raw_contexts)
comp_len = len(compressed.split())

print(f"\n Sıkıştırma Sonucu:")
print(f"   Önceki token sayısı : {raw_len}")
print(f"   Sonraki token sayısı: {comp_len}")
print(f"   Azalma oranı        : %{(1 - comp_len/raw_len)*100:.1f}")
print(f"\n--- Sıkıştırılmış Bağlam ---")
print(compressed[:400] + "...")

## Tam IntelliRAG Pipeline'ı

Dört modülü tek bir sınıfta birleştiriyoruz.

In [ ]:
# ── IntelliRAG Ana Sınıfı ────────────────────────────────────────────────────
class IntelliRAG:
    """
    Dört modülü birleştiren tam IntelliRAG pipeline'ı.

    Akış:
        Belge → Modül1 (Parçala) → Modül2 (Ara)
             → Modül3 (Sırala)  → Modül4 (Sıkıştır) → LLM
    """

    def __init__(
        self,
        retrieval_top_k: int = 10,
        rerank_top_k: int = 5,
        compression_ratio: float = 0.5
    ):
        print(" IntelliRAG başlatılıyor...\n")
        self.chunker    = SemanticChunker()
        self.retriever  = HybridRetriever()
        self.reranker   = CrossEncoderReranker(top_k=rerank_top_k)
        self.compressor = AdaptiveContextCompressor(compression_ratio=compression_ratio)
        self.retrieval_top_k = retrieval_top_k
        print("\n IntelliRAG tam hazır!")

    def index_documents(self, documents: List[str]):
        """Belge koleksiyonunu indeksle."""
        print(f"\n {len(documents)} belge indeksleniyor...")
        all_chunks = []
        for doc in tqdm(documents, desc="Parçalanıyor"):
            all_chunks.extend(self.chunker.chunk(doc))
        self.retriever.build_index(all_chunks)
        print(f" {len(all_chunks)} parça hazır.")

    def get_context(self, query: str) -> Dict:
        """Sorgu için en iyi bağlamı hazırla."""
        # Adım 1: Hibrit arama
        candidates = self.retriever.retrieve(query, top_k=self.retrieval_top_k)

        # Adım 2: Yeniden sırala
        reranked = self.reranker.rerank(query, candidates)

        # Adım 3: Sıkıştır
        contexts = [r['chunk'] for r in reranked]
        compressed = self.compressor.compress(query, contexts)

        return {
            'compressed_context': compressed,
            'raw_chunks': contexts,
            'scores': [r['ce_score'] for r in reranked]
        }

    def build_prompt(self, query: str, context: str) -> str:
        """LLM için prompt oluştur."""
        return (
            f"Bağlam bilgisine dayanarak soruyu kısa ve net yanıtla.\n\n"
            f"Bağlam:\n{context}\n\n"
            f"Soru: {query}\n\n"
            f"Cevap:"
        )


# ── Tam pipeline testi ───────────────────────────────────────────────────────
print("\n" + "="*60)
intellirag = IntelliRAG(retrieval_top_k=10, rerank_top_k=5, compression_ratio=0.5)

# Belgeleri indeksle
documents = [s['context'] for s in samples[:20]]
intellirag.index_documents(documents)

# Sorgu
query = samples[0]['question']
result = intellirag.get_context(query)
prompt = intellirag.build_prompt(query, result['compressed_context'])

print(f"\n{'='*60}")
print(f" Sorgu  : {query}")
print(f" Prompt (ilk 500 karakter):\n")
print(prompt[:500] + "...")

## Değerlendirme — RAGAS Metrikleri

| Metrik | Açıklama | Hedef |
|--------|----------|-------|
| **Faithfulness** | Cevap bağlama sadık mı? | ≥ 0.85 |
| **Context Precision** | Getirilen bağlam ne kadar alakalı? | ≥ 0.80 |
| **Answer Relevancy** | Cevap soruyu yanıtlıyor mu? | ≥ 0.82 |

In [ ]:
# ── Basit Kural Tabanlı Değerlendirme (RAGAS API olmadan) ────────────────────
# NOT: Gerçek RAGAS değerlendirmesi için OpenAI API anahtarı gerekir.
# Burada proxy metrikler kullanıyoruz.

def evaluate_context_precision(query: str, contexts: List[str], answer: str) -> float:
    """Bağlam hassasiyeti — cevap kelimelerinin bağlamda geçme oranı."""
    answer_words = set(answer.lower().split())
    context_text = ' '.join(contexts).lower()
    hits = sum(1 for w in answer_words if w in context_text)
    return hits / max(len(answer_words), 1)

def evaluate_answer_relevancy(query: str, answer: str, encoder) -> float:
    """Cevap alaka düzeyi — sorgu ve cevap benzerliği."""
    embs = encoder.encode([query, answer], convert_to_numpy=True)
    sim = np.dot(embs[0], embs[1]) / (
        np.linalg.norm(embs[0]) * np.linalg.norm(embs[1]) + 1e-8
    )
    return float(np.clip(sim, 0, 1))

def evaluate_faithfulness(answer: str, contexts: List[str]) -> float:
    """Sadakat — cevaptaki unigram'ların bağlamda geçme oranı."""
    answer_words = answer.lower().split()
    ctx = ' '.join(contexts).lower()
    supported = sum(1 for w in answer_words if w in ctx)
    return supported / max(len(answer_words), 1)


# ── Değerlendirme döngüsü ─────────────────────────────────────────────────────
print(" Değerlendirme başlıyor (ilk 20 örnek)...\n")

encoder = intellirag.retriever.encoder

metrics_intellirag = {'faithfulness': [], 'context_precision': [], 'answer_relevancy': []}
metrics_baseline   = {'faithfulness': [], 'context_precision': [], 'answer_relevancy': []}

for i, sample in enumerate(tqdm(samples[:20], desc="Değerlendiriliyor")):
    query   = sample['question']
    gt_ans  = sample['answer']

    # IntelliRAG bağlamı
    rag_result = intellirag.get_context(query)
    rag_ctx    = [rag_result['compressed_context']]

    # Baseline: ham bağlamın ilk parçası
    base_ctx = [sample['context'][:500]]

    # IntelliRAG metrikleri
    metrics_intellirag['faithfulness'].append(evaluate_faithfulness(gt_ans, rag_ctx))
    metrics_intellirag['context_precision'].append(evaluate_context_precision(query, rag_ctx, gt_ans))
    metrics_intellirag['answer_relevancy'].append(evaluate_answer_relevancy(query, gt_ans, encoder))

    # Baseline metrikleri
    metrics_baseline['faithfulness'].append(evaluate_faithfulness(gt_ans, base_ctx))
    metrics_baseline['context_precision'].append(evaluate_context_precision(query, base_ctx, gt_ans))
    metrics_baseline['answer_relevancy'].append(evaluate_answer_relevancy(query, gt_ans, encoder))


# ── Sonuçları göster ─────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"{'Metrik':<25} {'Baseline':>10} {'IntelliRAG':>12} {'Artış':>8}")
print("="*55)

for metric in ['faithfulness', 'context_precision', 'answer_relevancy']:
    base_val = np.mean(metrics_baseline[metric])
    rag_val  = np.mean(metrics_intellirag[metric])
    gain     = (rag_val - base_val) / base_val * 100
    label    = {'faithfulness': 'Sadakat', 'context_precision': 'Bağlam Hassasiyeti', 'answer_relevancy': 'Yanıt Alaka Düzeyi'}[metric]
    print(f"{label:<25} {base_val:>10.3f} {rag_val:>12.3f} {gain:>+7.1f}%")

print("="*55)
print(" Değerlendirme tamamlandı!")

## Özet

IntelliRAG'ın dört modülü:

| Modül | Yöntem | Kazanım |
|-------|--------|---------|
| Parçalama | Semantik eşik | Anlam bütünlüğü |
| Geri Alma | BM25 + FAISS | Kapsam genişliği |
| Yeniden Sıralama | Cross-Encoder | Hassas sıralama |
| Sıkıştırma | Kosinüs filtresi | Gürültü azaltma |

---
> **Bir sonraki notebook:** `02_IntelliRAG_Ablation.ipynb` — Ablation çalışması ve modül bazlı katkı analizi